# 04 — Mini Research: Your Own Question 迷你研究：你的問題

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewwangarchnycu/gis-open-data-workshop-2026/blob/main/notebooks/04_mini_research.ipynb)

Part of [Mapping the Unknown](https://github.com/andrewwangarchnycu/gis-open-data-workshop-2026) — feeds directly into the [One Map Challenge](../exercises/04_one_map_challenge/).

This notebook is a **template**. It runs end-to-end out of the box on the sample dataset, and every `# TODO` marks where you adapt it to your own research question.
本筆記本是一份**範本**。它可直接以範例資料集完整執行，每個 `# TODO` 標示你需要依自己研究問題調整之處。

In [ ]:
!pip install geopandas shapely matplotlib -q

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import matplotlib.pyplot as plt

## 0 — State your question 陳述你的問題

Fill this in before writing any code — see [Lesson 01](../lessons/01-mapping-the-unknown/) and [Lesson 03](../lessons/03-open-geospatial-data/).
在寫任何程式碼之前先填寫這裡。

- **My question 我的問題**: _(e.g. Where are the greener public spaces?)_
- **Required variables 所需變數**: _______
- **Dataset(s) I'll use 我將使用的資料集**: _______ (sample data below, or your own GeoJSON — see note in Step 1)

## 1 — Load 載入

Using the workshop sample dataset by default. To use your own open data instead, replace this cell with `gpd.read_file("https://.../your_data.geojson")` — read directly from a URL so the notebook stays independent of any local filesystem.
預設使用工作坊範例資料集。若要改用你自己的開放資料，將此儲存格改為 `gpd.read_file("https://.../your_data.geojson")`——直接從網址讀取，讓筆記本不依賴任何本機檔案系統。

In [ ]:
tree_coords = [
    (121.5320, 25.0335), (121.5325, 25.0338), (121.5330, 25.0330),
    (121.5340, 25.0345), (121.5345, 25.0348), (121.5300, 25.0320),
    (121.5305, 25.0322), (121.5360, 25.0360),
]
trees = gpd.GeoDataFrame({"tree_id": range(1, len(tree_coords) + 1)},
                          geometry=[Point(xy) for xy in tree_coords], crs="EPSG:4326")

public_spaces = gpd.GeoDataFrame(
    {"name": ["Riverside Park", "Central Plaza", "Corner Lot Square"]},
    geometry=[
        Polygon([(121.5315, 25.0328), (121.5335, 25.0328), (121.5335, 25.0350), (121.5315, 25.0350)]),
        Polygon([(121.5295, 25.0315), (121.5310, 25.0315), (121.5310, 25.0328), (121.5295, 25.0328)]),
        Polygon([(121.5352, 25.0352), (121.5365, 25.0352), (121.5365, 25.0365), (121.5352, 25.0365)]),
    ],
    crs="EPSG:4326",
)
# TODO: replace the two GeoDataFrames above with your own data if you have it

## 2 — Inspect 檢視

In [ ]:
print(trees.crs, public_spaces.crs)
print(trees.shape, public_spaces.shape)
trees.head()

## 3 — Clean 清理

In [ ]:
trees = trees[trees.geometry.notnull()].copy()
public_spaces = public_spaces[public_spaces.geometry.notnull()].copy()
# TODO: add any cleaning your own dataset needs (e.g. dropna on a specific attribute column)

## 4 — Spatial Operation 空間運算

**Why 為什麼**: _(fill in — e.g. "buffer trees by 10m because we're asking which spaces are within reach of a tree")_

In [ ]:
trees_m = trees.to_crs("EPSG:3826")
public_spaces_m = public_spaces.to_crs("EPSG:3826")

BUFFER_METERS = 10  # TODO: change to match your research question
tree_buffers = trees_m.copy()
tree_buffers["geometry"] = tree_buffers.buffer(BUFFER_METERS)

joined = gpd.sjoin(trees_m, public_spaces_m, how="left", predicate="within")
tree_counts = joined.groupby("name").size().rename("tree_count")
public_spaces_m = public_spaces_m.merge(tree_counts, on="name", how="left")
public_spaces_m["tree_count"] = public_spaces_m["tree_count"].fillna(0)

## 5 — Calculate 計算

In [ ]:
public_spaces_m["area_m2"] = public_spaces_m.geometry.area
public_spaces_m["tree_density_per_1000m2"] = (
    public_spaces_m["tree_count"] / public_spaces_m["area_m2"] * 1000
)
public_spaces_m[["name", "tree_count", "area_m2", "tree_density_per_1000m2"]]

## 6 — Visualize 視覺化

This figure is your **evidence**, not your final map — you'll redesign it as a proper research map in [Lesson 07](../lessons/07-research-map-design/) and the [One Map Challenge](../exercises/04_one_map_challenge/).
這張圖是你的**證據**，還不是最終地圖——你會在[課程 07](../lessons/07-research-map-design/)與[一張地圖挑戰](../exercises/04_one_map_challenge/)中將它重新設計為正式研究地圖。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
public_spaces_m.plot(ax=ax, column="tree_density_per_1000m2", cmap="Greens",
                      edgecolor="black", legend=True)
trees_m.plot(ax=ax, color="black", markersize=15)
ax.set_title("My mini research map (draft) 我的迷你研究地圖（草稿）")
ax.set_axis_off()
plt.show()

## Research interpretation exercise 研究詮釋練習

Fill in the What/Where/Why/So-what chain from [Lesson 06](../lessons/06-spatial-insight/):

- **What? 是什麼？** _______
- **Where? 在哪裡？** _______
- **Why? 為什麼？** _______
- **So what? 所以呢？** _______

Carry this straight into the [One Map Challenge](../exercises/04_one_map_challenge/).

Prefer to start from real data instead of your own sample? [`case-studies/`](../case-studies/) has three ready-made real Taiwan datasets (trees, temperature, bus positions) you can adapt directly. 想直接從真實資料開始，而非自建範例？[`case-studies/`](../case-studies/) 提供三組現成的真實台灣資料集（樹木、氣溫、公車位置）可直接調整使用。